# LightRAG Hybrid Search 실사용 데모
## Vector Only vs Vector + BM25 하이브리드 쿼리 비교

이 노트북은 실제 LightRAG 객체를 생성하고, 문서를 삽입하여 지식그래프를 구축한 뒤,
**BM25 하이브리드 검색 ON/OFF 시 쿼리 결과를 비교**합니다.

### 시나리오
1. LLM 엔드포인트 설정 (OpenAI 호환 API)
2. LightRAG 인스턴스 2개 생성 (Vector Only / Hybrid)
3. `ainsert_custom_kg`로 동일한 지식그래프 삽입
4. 동일 쿼리에 대해 `aquery_data`로 검색 결과 비교
5. 고유명사, 기술용어, 서술형 쿼리별 차이 분석
6. (보너스) `ainsert`로 실제 LLM 엔티티 추출 파이프라인 데모

## 1. 환경 설정 및 LLM 엔드포인트 구성

OpenAI 호환 API 엔드포인트를 사용하여 LLM과 임베딩 함수를 설정합니다.

- **LLM**: `qwen-task-pool` 모델 (vLLM 서버)
- **Embedding**: 간이 단어 빈도 기반 임베딩 (별도 임베딩 모델 없이 동작)

> **참고**: LLM 엔드포인트가 연결 불가능한 환경에서도 `ainsert_custom_kg` + `aquery_data(mode="naive")`는 정상 동작합니다.

In [ ]:
import sys
import os
import shutil
import asyncio
import numpy as np

sys.path.insert(0, '..')

from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc, Tokenizer, TokenizerInterface
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.bm25_index import BM25Index, reciprocal_rank_fusion

# ============================================================
# LLM 엔드포인트 설정 (OpenAI 호환 API)
# ============================================================
LLM_BASE_URL = "http://222.117.133.162:30010/v1"
LLM_MODEL = "qwen-task-pool"
LLM_API_KEY = "asdf"

async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    """OpenAI 호환 LLM 함수 (vLLM 서버 사용)"""
    try:
        return await openai_complete_if_cache(
            LLM_MODEL,
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages,
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
            **kwargs,
        )
    except Exception as e:
        # LLM 연결 실패 시 fallback (오프라인 환경 대응)
        if 'low_level_keywords' in prompt or 'high_level_keywords' in prompt:
            return '{"low_level_keywords": ["test"], "high_level_keywords": ["test"]}'
        return f'[LLM 연결 불가: {type(e).__name__}]'


# ============================================================
# 임베딩 함수 설정 (간이 단어 빈도 기반)
# ============================================================
class SimpleTokenizer(TokenizerInterface):
    """간단한 토크나이저 (데모용)"""
    def encode(self, content: str):
        return content.split()
    def decode(self, tokens):
        return ' '.join(tokens)


VOCAB = {}
EMB_DIM = 64

def _word_to_idx(word: str) -> int:
    w = word.lower().strip()
    if w not in VOCAB:
        VOCAB[w] = len(VOCAB)
    return VOCAB[w]

async def mock_embedding(texts: list[str]) -> np.ndarray:
    """단어 빈도 기반 임베딩 (의미 유사도가 아닌 단어 겹침 기반)"""
    result = []
    for text in texts:
        vec = np.zeros(EMB_DIM)
        for word in text.split():
            idx = _word_to_idx(word) % EMB_DIM
            vec[idx] += 1.0
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        result.append(vec)
    return np.array(result)


print('환경 설정 완료')
print(f'  LLM 엔드포인트: {LLM_BASE_URL}')
print(f'  LLM 모델:       {LLM_MODEL}')
print(f'  임베딩 차원:     {EMB_DIM}')

## 2. 샘플 지식그래프 데이터 준비

AI/ML 도메인의 기술 문서를 시뮬레이션하는 청크, 엔티티, 관계를 준비합니다.

In [2]:
# 샘플 문서 청크 (실제 기술 문서에서 발췌한 것처럼 구성)
SAMPLE_CHUNKS = [
    {
        "content": "GPT-4o is a multimodal large language model developed by OpenAI. It can process text, images, and audio inputs simultaneously. GPT-4o achieves state-of-the-art performance on multiple benchmarks including MMLU and HumanEval.",
        "source_id": "doc_openai",
        "file_path": "ai_models_survey.txt",
    },
    {
        "content": "LightRAG is a graph-based Retrieval Augmented Generation system developed by HKUDS research group. It constructs knowledge graphs from documents and uses them for more structured retrieval compared to traditional RAG systems.",
        "source_id": "doc_lightrag",
        "file_path": "rag_systems_review.txt",
    },
    {
        "content": "BM25 (Best Matching 25) is a probabilistic ranking function widely used in information retrieval. It scores documents based on term frequency (TF) and inverse document frequency (IDF). BM25 is the default ranking algorithm in Elasticsearch and Apache Lucene.",
        "source_id": "doc_ir",
        "file_path": "information_retrieval.txt",
    },
    {
        "content": "Vector similarity search uses dense embeddings to find semantically similar documents. Common approaches include cosine similarity and approximate nearest neighbor (ANN) algorithms like HNSW and IVF.",
        "source_id": "doc_vector",
        "file_path": "search_methods.txt",
    },
    {
        "content": "Reciprocal Rank Fusion (RRF) is a method for combining search results from multiple retrieval systems. The formula is RRF(d) = sum(1/(k+rank)) where k is typically 60. RRF is used in hybrid search systems that combine keyword and semantic search.",
        "source_id": "doc_rrf",
        "file_path": "fusion_methods.txt",
    },
    {
        "content": "BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI. It introduced the masked language model pre-training objective and achieved breakthrough results on the GLUE benchmark.",
        "source_id": "doc_bert",
        "file_path": "ai_models_survey.txt",
    },
    {
        "content": "Knowledge graph construction involves entity extraction, relation extraction, and graph building. Named Entity Recognition (NER) and coreference resolution are key components of the entity extraction pipeline.",
        "source_id": "doc_kg",
        "file_path": "kg_construction.txt",
    },
    {
        "content": "The HKUDS research group at the University of Hong Kong focuses on data science and systems research. Their notable projects include LightRAG for graph-based RAG and various recommender system frameworks.",
        "source_id": "doc_hkuds",
        "file_path": "research_groups.txt",
    },
]

# 엔티티 정의
SAMPLE_ENTITIES = [
    {"entity_name": "GPT-4O", "entity_type": "AI_MODEL", "description": "OpenAI가 개발한 멀티모달 대규모 언어 모델. 텍스트, 이미지, 오디오를 동시 처리 가능.", "source_id": "doc_openai", "file_path": "ai_models_survey.txt"},
    {"entity_name": "OPENAI", "entity_type": "ORGANIZATION", "description": "GPT 시리즈를 개발한 AI 연구 기업.", "source_id": "doc_openai", "file_path": "ai_models_survey.txt"},
    {"entity_name": "LIGHTRAG", "entity_type": "SOFTWARE", "description": "HKUDS가 개발한 그래프 기반 RAG 시스템. 지식그래프를 활용한 구조적 검색 제공.", "source_id": "doc_lightrag", "file_path": "rag_systems_review.txt"},
    {"entity_name": "HKUDS", "entity_type": "ORGANIZATION", "description": "홍콩대학교 데이터사이언스 연구 그룹. LightRAG 등 다수 프로젝트 수행.", "source_id": "doc_hkuds", "file_path": "research_groups.txt"},
    {"entity_name": "BM25", "entity_type": "ALGORITHM", "description": "정보 검색에 사용되는 확률적 랭킹 함수. TF-IDF 기반으로 문서 점수를 계산.", "source_id": "doc_ir", "file_path": "information_retrieval.txt"},
    {"entity_name": "ELASTICSEARCH", "entity_type": "SOFTWARE", "description": "오픈소스 분산 검색 엔진. BM25를 기본 랭킹 알고리즘으로 사용.", "source_id": "doc_ir", "file_path": "information_retrieval.txt"},
    {"entity_name": "BERT", "entity_type": "AI_MODEL", "description": "Google AI가 개발한 양방향 트랜스포머 모델. GLUE 벤치마크에서 획기적 성과 달성.", "source_id": "doc_bert", "file_path": "ai_models_survey.txt"},
    {"entity_name": "GOOGLE_AI", "entity_type": "ORGANIZATION", "description": "Google의 AI 연구 부문. BERT, Transformer 등 핵심 모델 개발.", "source_id": "doc_bert", "file_path": "ai_models_survey.txt"},
    {"entity_name": "RRF", "entity_type": "ALGORITHM", "description": "Reciprocal Rank Fusion. 여러 검색 시스템의 결과를 순위 기반으로 병합하는 방법.", "source_id": "doc_rrf", "file_path": "fusion_methods.txt"},
    {"entity_name": "VECTOR_SEARCH", "entity_type": "TECHNIQUE", "description": "Dense embedding을 사용한 의미 기반 유사 문서 검색. HNSW, IVF 등의 ANN 알고리즘 활용.", "source_id": "doc_vector", "file_path": "search_methods.txt"},
    {"entity_name": "KNOWLEDGE_GRAPH", "entity_type": "TECHNIQUE", "description": "엔티티와 관계를 노드와 엣지로 표현하는 그래프 구조. NER과 공참조 해결이 핵심 구성 요소.", "source_id": "doc_kg", "file_path": "kg_construction.txt"},
]

# 관계 정의
SAMPLE_RELATIONSHIPS = [
    {"src_id": "OPENAI", "tgt_id": "GPT-4O", "description": "OpenAI가 GPT-4o를 개발함", "keywords": "개발, 생성", "weight": 2.0, "source_id": "doc_openai", "file_path": "ai_models_survey.txt"},
    {"src_id": "HKUDS", "tgt_id": "LIGHTRAG", "description": "HKUDS 연구 그룹이 LightRAG을 개발함", "keywords": "개발, 연구", "weight": 2.0, "source_id": "doc_lightrag", "file_path": "rag_systems_review.txt"},
    {"src_id": "LIGHTRAG", "tgt_id": "KNOWLEDGE_GRAPH", "description": "LightRAG은 지식그래프를 활용하여 검색 수행", "keywords": "활용, 검색", "weight": 1.5, "source_id": "doc_lightrag", "file_path": "rag_systems_review.txt"},
    {"src_id": "ELASTICSEARCH", "tgt_id": "BM25", "description": "Elasticsearch가 BM25를 기본 랭킹 알고리즘으로 사용", "keywords": "사용, 랭킹", "weight": 1.5, "source_id": "doc_ir", "file_path": "information_retrieval.txt"},
    {"src_id": "GOOGLE_AI", "tgt_id": "BERT", "description": "Google AI가 BERT를 개발함", "keywords": "개발, NLP", "weight": 2.0, "source_id": "doc_bert", "file_path": "ai_models_survey.txt"},
    {"src_id": "RRF", "tgt_id": "VECTOR_SEARCH", "description": "RRF가 벡터 검색 결과를 BM25 결과와 병합", "keywords": "병합, 하이브리드", "weight": 1.5, "source_id": "doc_rrf", "file_path": "fusion_methods.txt"},
    {"src_id": "RRF", "tgt_id": "BM25", "description": "RRF가 BM25 결과를 벡터 검색 결과와 병합", "keywords": "병합, 하이브리드", "weight": 1.5, "source_id": "doc_rrf", "file_path": "fusion_methods.txt"},
    {"src_id": "GPT-4O", "tgt_id": "BERT", "description": "GPT-4o와 BERT는 모두 트랜스포머 기반 언어 모델", "keywords": "트랜스포머, 언어모델", "weight": 1.0, "source_id": "doc_openai", "file_path": "ai_models_survey.txt"},
]

CUSTOM_KG = {
    "chunks": SAMPLE_CHUNKS,
    "entities": SAMPLE_ENTITIES,
    "relationships": SAMPLE_RELATIONSHIPS,
}

print(f"준비된 데이터:")
print(f"  청크: {len(SAMPLE_CHUNKS)}개")
print(f"  엔티티: {len(SAMPLE_ENTITIES)}개")
print(f"  관계: {len(SAMPLE_RELATIONSHIPS)}개")

준비된 데이터:
  청크: 8개
  엔티티: 11개
  관계: 8개


## 3. LightRAG 인스턴스 생성 (Vector Only vs Hybrid)

동일한 데이터를 넣되, 하나는 `enable_hybrid_search=False`, 다른 하나는 `True`로 설정합니다.
두 인스턴스 모두 **동일한 LLM 엔드포인트**를 사용합니다.

In [ ]:
WORK_DIR_VECTOR = '/tmp/lightrag_demo_vector'
WORK_DIR_HYBRID = '/tmp/lightrag_demo_hybrid'

# 이전 데이터 정리
for d in [WORK_DIR_VECTOR, WORK_DIR_HYBRID]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

embedding_func = EmbeddingFunc(
    embedding_dim=EMB_DIM,
    max_token_size=4096,
    func=mock_embedding,
)

tokenizer = Tokenizer('simple-tokenizer', SimpleTokenizer())

# --- 1) Vector Only RAG ---
rag_vector = LightRAG(
    working_dir=WORK_DIR_VECTOR,
    llm_model_func=llm_model_func,  # 실제 LLM 엔드포인트 사용
    embedding_func=embedding_func,
    tokenizer=tokenizer,
    addon_params={
        'enable_hybrid_search': False,  # 기존 방식: 벡터 검색만 사용
    },
)

# --- 2) Hybrid RAG (Vector + BM25) ---
rag_hybrid = LightRAG(
    working_dir=WORK_DIR_HYBRID,
    llm_model_func=llm_model_func,  # 동일한 LLM 엔드포인트
    embedding_func=embedding_func,
    tokenizer=tokenizer,
    addon_params={
        'enable_hybrid_search': True,  # 새 방식: 벡터 + BM25 하이브리드
    },
)

print('LightRAG 인스턴스 생성 완료')
print(f'  LLM 엔드포인트: {LLM_BASE_URL} (모델: {LLM_MODEL})')
print(f'  Vector Only: enable_hybrid_search = {rag_vector._addon_params.get("enable_hybrid_search")}')
print(f'  Hybrid:      enable_hybrid_search = {rag_hybrid._addon_params.get("enable_hybrid_search")}')

## 4. 스토리지 초기화 및 지식그래프 삽입

In [4]:
# 스토리지 초기화
await rag_vector.initialize_storages()
await rag_hybrid.initialize_storages()

# 동일한 지식그래프 데이터 삽입 (ainsert_custom_kg는 LLM 호출 없이 직접 KG 삽입)
await rag_vector.ainsert_custom_kg(CUSTOM_KG)
await rag_hybrid.ainsert_custom_kg(CUSTOM_KG)

print('지식그래프 삽입 완료!')
print()

# BM25 인덱스 상태 확인
print('--- BM25 인덱스 상태 ---')
print(f'Vector Only RAG:')
print(f'  _bm25_chunks:   {"Built" if rag_vector._bm25_chunks and rag_vector._bm25_chunks.is_built else "None"}')
print(f'  _bm25_entities: {"Built" if rag_vector._bm25_entities and rag_vector._bm25_entities.is_built else "None"}')
print(f'  _bm25_relations:{"Built" if rag_vector._bm25_relations and rag_vector._bm25_relations.is_built else "None"}')
print()
print(f'Hybrid RAG:')
print(f'  _bm25_chunks:   {"Built" if rag_hybrid._bm25_chunks and rag_hybrid._bm25_chunks.is_built else "None"}')
print(f'  _bm25_entities: {"Built" if rag_hybrid._bm25_entities and rag_hybrid._bm25_entities.is_built else "None"}')
print(f'  _bm25_relations:{"Built" if rag_hybrid._bm25_relations and rag_hybrid._bm25_relations.is_built else "None"}')
print(f'  _bm25_stale:    {rag_hybrid._bm25_stale}')

INFO: [] Process 16176 KV load full_docs with 0 records


INFO: [] Process 16176 KV load text_chunks with 0 records


INFO: [] Process 16176 KV load full_entities with 0 records


INFO: [] Process 16176 KV load full_relations with 0 records


INFO: [] Process 16176 KV load entity_chunks with 0 records


...

INFO: [] chunks flush: embedding 8 vectors in 1 batch(es) (batch_num=10)


INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)


INFO: [] Writing graph with 11 nodes, 8 edges


INFO: In memory DB persist to disk


지식그래프 삽입 완료!

--- BM25 인덱스 상태 ---
Vector Only RAG:
  _bm25_chunks:   None
  _bm25_entities: None
  _bm25_relations:None

Hybrid RAG:
  _bm25_chunks:   None
  _bm25_entities: None
  _bm25_relations:None
  _bm25_stale:    True


## 5. BM25 인덱스 직접 확인

Hybrid RAG에서 BM25 인덱스가 실제로 어떤 결과를 반환하는지 직접 확인합니다.

In [5]:
# Hybrid RAG의 BM25 인덱스 lazy build 트리거
# (aquery_data 호출 시 자동으로 빌드되지만, 먼저 직접 확인)
await rag_hybrid._build_bm25_indices()

print('=== BM25 엔티티 인덱스 테스트 ===')
test_queries = ['GPT-4o', 'HKUDS', 'BM25', 'transformer language model']

for q in test_queries:
    results = rag_hybrid._bm25_entities.query(q, top_k=3)
    print(f'\n쿼리: "{q}"')
    if results:
        for r in results:
            print(f'  -> {r["id"]:20s} (BM25 score: {r["score"]:.4f})')
    else:
        print('  -> (결과 없음)')

print('\n=== BM25 청크 인덱스 테스트 ===')
for q in ['GPT-4o multimodal', 'HKUDS LightRAG', 'Elasticsearch BM25']:
    results = rag_hybrid._bm25_chunks.query(q, top_k=2)
    print(f'\n쿼리: "{q}"')
    if results:
        for r in results:
            print(f'  -> chunk {r["id"][:20]}... (BM25 score: {r["score"]:.4f})')
    else:
        print('  -> (결과 없음)')

INFO: BM25 index built with 8 documents


INFO: BM25 index built with 11 documents


INFO: BM25 index built with 8 documents


INFO: BM25 indices built: chunks=8, entities=11, relations=8


=== BM25 엔티티 인덱스 테스트 ===

쿼리: "GPT-4o"
  -> GPT-4O               (BM25 score: 3.0293)
  -> OPENAI               (BM25 score: 1.6350)

쿼리: "HKUDS"
  -> HKUDS                (BM25 score: 2.0906)

쿼리: "BM25"
  -> BM25                 (BM25 score: 1.8621)

쿼리: "transformer language model"
  -> GOOGLE_AI            (BM25 score: 2.0085)

=== BM25 청크 인덱스 테스트 ===

쿼리: "GPT-4o multimodal"
  -> chunk chunk-498ef3524ff4f2... (BM25 score: 6.0257)

쿼리: "HKUDS LightRAG"
  -> chunk chunk-c8f100f5c4c6b6... (BM25 score: 1.9449)
  -> chunk chunk-64ebde6e16b3f9... (BM25 score: 1.9177)

쿼리: "Elasticsearch BM25"
  -> chunk chunk-f1e105c34c6ff8... (BM25 score: 3.6645)


## 6. 쿼리 비교: Vector Only vs Hybrid

핵심 비교 테스트입니다. 동일한 쿼리를 두 RAG 인스턴스에 날려서
검색된 엔티티, 관계, 청크를 비교합니다.

`aquery_data`는 LLM 응답 생성 없이 **검색 결과만** 반환하므로 순수한 검색 성능 비교가 가능합니다.

In [6]:
async def compare_query(query: str, mode: str = 'naive', description: str = ''):
    """Vector Only vs Hybrid 쿼리 결과 비교"""
    print('=' * 70)
    print(f'쿼리: "{query}"')
    print(f'모드: {mode} | {description}')
    print('=' * 70)
    
    param = QueryParam(mode=mode, top_k=5)
    
    # Vector Only 쿼리
    result_vec = await rag_vector.aquery_data(query, param=param)
    # Hybrid 쿼리
    result_hyb = await rag_hybrid.aquery_data(query, param=param)
    
    def extract_info(result):
        if result.get('status') != 'success':
            return {'entities': [], 'relationships': [], 'chunks': []}
        data = result.get('data', {})
        return {
            'entities': [e.get('entity_name', '?') for e in data.get('entities', [])],
            'relationships': [(r.get('src_id','?'), r.get('tgt_id','?')) for r in data.get('relationships', [])],
            'chunks': [c.get('content', '')[:60] + '...' for c in data.get('chunks', [])],
        }
    
    vec_info = extract_info(result_vec)
    hyb_info = extract_info(result_hyb)
    
    print(f'\n--- [Vector Only] ---')
    print(f'  엔티티 ({len(vec_info["entities"])}개): {vec_info["entities"]}')
    print(f'  관계 ({len(vec_info["relationships"])}개): {vec_info["relationships"][:3]}')
    print(f'  청크 ({len(vec_info["chunks"])}개):')
    for c in vec_info['chunks'][:3]:
        print(f'    - {c}')
    
    print(f'\n--- [Hybrid: Vector + BM25] ---')
    print(f'  엔티티 ({len(hyb_info["entities"])}개): {hyb_info["entities"]}')
    print(f'  관계 ({len(hyb_info["relationships"])}개): {hyb_info["relationships"][:3]}')
    print(f'  청크 ({len(hyb_info["chunks"])}개):')
    for c in hyb_info['chunks'][:3]:
        print(f'    - {c}')
    
    # 차이점 분석
    vec_ents = set(vec_info['entities'])
    hyb_ents = set(hyb_info['entities'])
    only_in_hybrid = hyb_ents - vec_ents
    only_in_vector = vec_ents - hyb_ents
    
    if only_in_hybrid:
        print(f'\n  [BM25가 추가로 찾은 엔티티]: {only_in_hybrid}')
    if only_in_vector:
        print(f'  [Vector에서만 찾은 엔티티]: {only_in_vector}')
    if not only_in_hybrid and not only_in_vector:
        print(f'\n  [결과 동일] - 두 방식 모두 같은 엔티티를 검색')
    
    print()

In [7]:
# 테스트 1: 고유명사 쿼리 (BM25 강점)
await compare_query(
    'GPT-4o', 
    mode='naive',
    description='고유명사 검색 - BM25가 정확한 토큰 매칭으로 강점을 보이는 케이스'
)

INFO: [] Process 16176 reloading chunks due to update by another process


INFO:nano-vectordb:Load (8, 64) data


INFO:nano-vectordb:Init {'embedding_dim': 64, 'metric': 'cosine', 'storage_file': '/tmp/lightrag_demo_vector/vdb_chunks.json'} 8 data


INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks


INFO: Hybrid chunk search: 1 results after RRF


INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks


쿼리: "GPT-4o"
모드: naive | 고유명사 검색 - BM25가 정확한 토큰 매칭으로 강점을 보이는 케이스

--- [Vector Only] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - GPT-4o is a multimodal large language model developed by Ope...

--- [Hybrid: Vector + BM25] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - GPT-4o is a multimodal large language model developed by Ope...

  [결과 동일] - 두 방식 모두 같은 엔티티를 검색



In [8]:
# 테스트 2: 기관명 쿼리 (BM25 강점)
await compare_query(
    'HKUDS', 
    mode='naive',
    description='기관/그룹명 검색 - 고유 식별자를 BM25가 정확히 매칭'
)

INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks


INFO: Hybrid chunk search: 2 results after RRF


쿼리: "HKUDS"
모드: naive | 기관/그룹명 검색 - 고유 식별자를 BM25가 정확히 매칭


INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks



--- [Vector Only] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - The HKUDS research group at the University of Hong Kong focu...

--- [Hybrid: Vector + BM25] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - The HKUDS research group at the University of Hong Kong focu...

  [결과 동일] - 두 방식 모두 같은 엔티티를 검색



In [9]:
# 테스트 3: 기술 용어 쿼리 (BM25 강점)
await compare_query(
    'BM25 ranking algorithm', 
    mode='naive',
    description='기술 용어 검색 - BM25가 정확한 키워드 포함 문서를 우선 반환'
)

INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks


INFO: Hybrid chunk search: 1 results after RRF


INFO: Naive query: 1 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 1 chunks


쿼리: "BM25 ranking algorithm"
모드: naive | 기술 용어 검색 - BM25가 정확한 키워드 포함 문서를 우선 반환

--- [Vector Only] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - BM25 (Best Matching 25) is a probabilistic ranking function ...

--- [Hybrid: Vector + BM25] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (1개):
    - BM25 (Best Matching 25) is a probabilistic ranking function ...

  [결과 동일] - 두 방식 모두 같은 엔티티를 검색



In [10]:
# 테스트 4: 서술형 쿼리 (Vector 강점)
await compare_query(
    'How do AI systems understand and process natural language?', 
    mode='naive',
    description='서술형 쿼리 - 벡터 검색이 의미 기반 매칭으로 강점을 보이는 케이스'
)

INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 6 chunks


INFO: Hybrid chunk search: 8 results after RRF


INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 6 chunks


쿼리: "How do AI systems understand and process natural language?"
모드: naive | 서술형 쿼리 - 벡터 검색이 의미 기반 매칭으로 강점을 보이는 케이스

--- [Vector Only] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (6개):
    - LightRAG is a graph-based Retrieval Augmented Generation sys...
    - Reciprocal Rank Fusion (RRF) is a method for combining searc...
    - The HKUDS research group at the University of Hong Kong focu...

--- [Hybrid: Vector + BM25] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (6개):
    - LightRAG is a graph-based Retrieval Augmented Generation sys...
    - Reciprocal Rank Fusion (RRF) is a method for combining searc...
    - The HKUDS research group at the University of Hong Kong focu...

  [결과 동일] - 두 방식 모두 같은 엔티티를 검색



In [11]:
# 테스트 5: 복합 쿼리 (하이브리드 강점)
await compare_query(
    'Elasticsearch uses BM25 for document retrieval', 
    mode='naive',
    description='복합 쿼리 - 키워드(Elasticsearch, BM25) + 의미(retrieval) 모두 포함'
)

INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 4 chunks


INFO: Hybrid chunk search: 5 results after RRF


INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)


INFO: Final context: 4 chunks


쿼리: "Elasticsearch uses BM25 for document retrieval"
모드: naive | 복합 쿼리 - 키워드(Elasticsearch, BM25) + 의미(retrieval) 모두 포함

--- [Vector Only] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (4개):
    - LightRAG is a graph-based Retrieval Augmented Generation sys...
    - Reciprocal Rank Fusion (RRF) is a method for combining searc...
    - BM25 (Best Matching 25) is a probabilistic ranking function ...

--- [Hybrid: Vector + BM25] ---
  엔티티 (0개): []
  관계 (0개): []
  청크 (4개):
    - LightRAG is a graph-based Retrieval Augmented Generation sys...
    - BM25 (Best Matching 25) is a probabilistic ranking function ...
    - Reciprocal Rank Fusion (RRF) is a method for combining searc...

  [결과 동일] - 두 방식 모두 같은 엔티티를 검색



## 7. BM25 인덱스 모듈 단위 테스트

BM25Index 클래스와 RRF 함수의 동작을 직접 검증합니다.

In [12]:
print('=== BM25Index 단위 테스트 ===')
print()

# 1. 빌드 및 기본 쿼리
idx = BM25Index()
idx.build({
    'd1': 'machine learning deep neural network training',
    'd2': 'natural language processing text generation',
    'd3': 'computer vision image recognition deep learning',
    'd4': 'reinforcement learning agent reward policy',
    'd5': 'transfer learning domain adaptation fine tuning',
    'd6': 'graph neural network node classification',
})
results = idx.query('machine learning', top_k=3)
print(f'테스트 1 - 기본 쿼리: {[r["id"] for r in results]}')
assert len(results) > 0 and results[0]['id'] == 'd1'
print('  통과!')

# 2. 빈 인덱스
empty = BM25Index()
assert not empty.is_built
assert empty.query('test') == []
print('테스트 2 - 빈 인덱스: 통과!')

# 3. 빈 쿼리
assert idx.query('') == []
print('테스트 3 - 빈 쿼리: 통과!')

# 4. 매칭 없는 쿼리
assert idx.query('xyznonexistent') == []
print('테스트 4 - 매칭 없는 쿼리: 통과!')

print()
print('=== RRF 함수 단위 테스트 ===')
print()

# 5. 기본 RRF 병합
vec = [{'id': 'a', 's': 0.9}, {'id': 'b', 's': 0.8}]
bm = [{'id': 'b', 's': 5.0}, {'id': 'c', 's': 3.0}]
fused = reciprocal_rank_fusion(vec, bm)
ids = [r['id'] for r in fused]
assert ids[0] == 'b'  # 양쪽 모두에 있는 'b'가 1등
assert set(ids) == {'a', 'b', 'c'}
print(f'테스트 5 - 기본 RRF: {ids}')
print(f'  "b"가 양쪽에 모두 등장 -> 1등. 통과!')

# 6. 빈 입력
assert reciprocal_rank_fusion([], []) == []
print('테스트 6 - 빈 입력 RRF: 통과!')

# 7. 한쪽만 있는 경우
fused = reciprocal_rank_fusion([{'id': 'x'}, {'id': 'y'}], [])
assert [r['id'] for r in fused] == ['x', 'y']
print('테스트 7 - 한쪽만 있는 RRF: 통과!')

print()
print('모든 단위 테스트 통과!')

INFO: BM25 index built with 6 documents


=== BM25Index 단위 테스트 ===

테스트 1 - 기본 쿼리: ['d1', 'd4', 'd3']
  통과!
테스트 2 - 빈 인덱스: 통과!
테스트 3 - 빈 쿼리: 통과!
테스트 4 - 매칭 없는 쿼리: 통과!

=== RRF 함수 단위 테스트 ===

테스트 5 - 기본 RRF: ['b', 'a', 'c']
  "b"가 양쪽에 모두 등장 -> 1등. 통과!
테스트 6 - 빈 입력 RRF: 통과!
테스트 7 - 한쪽만 있는 RRF: 통과!

모든 단위 테스트 통과!


## 8. RRF 점수 계산 시각화

In [13]:
import pandas as pd

# 시뮬레이션 데이터
vector_results = [
    {'id': 'GPT-4O', 'score': 0.92},
    {'id': 'BERT', 'score': 0.88},
    {'id': 'LIGHTRAG', 'score': 0.81},
    {'id': 'VECTOR_SEARCH', 'score': 0.75},
]

bm25_results = [
    {'id': 'LIGHTRAG', 'score': 8.5},   # LightRAG 문서에 키워드 다수 포함
    {'id': 'HKUDS', 'score': 6.2},       # HKUDS가 BM25에서만 검출
    {'id': 'GPT-4O', 'score': 4.1},
    {'id': 'BM25', 'score': 3.8},        # BM25 알고리즘 자체 문서
]

k = 60
all_ids = set()
vec_rank = {}
bm25_rank = {}

for i, r in enumerate(vector_results, 1):
    vec_rank[r['id']] = i
    all_ids.add(r['id'])
for i, r in enumerate(bm25_results, 1):
    bm25_rank[r['id']] = i
    all_ids.add(r['id'])

rows = []
for doc in all_ids:
    vr = vec_rank.get(doc)
    br = bm25_rank.get(doc)
    vs = 1/(k+vr) if vr else 0
    bs = 1/(k+br) if br else 0
    rows.append({
        '엔티티': doc,
        'Vector 순위': vr or '-',
        'BM25 순위': br or '-',
        'Vector RRF': f'{vs:.6f}' if vr else '-',
        'BM25 RRF': f'{bs:.6f}' if br else '-',
        '합산 RRF': f'{vs+bs:.6f}',
        '출처': 'BOTH' if (vr and br) else ('Vector' if vr else 'BM25'),
    })

df = pd.DataFrame(rows).sort_values('합산 RRF', ascending=False).reset_index(drop=True)
df.index = df.index + 1
df.index.name = '최종 순위'
print('RRF 점수 계산 상세 (k=60)')
print()
df

RRF 점수 계산 상세 (k=60)



,엔티티,Vector 순위,BM25 순위,Vector RRF,BM25 RRF,합산 RRF,출처
최종 순위,,,,,,,
1,GPT-4O,1,3,0.016393,0.015873,0.032266,BOTH
2,LIGHTRAG,3,1,0.015873,0.016393,0.032266,BOTH
3,BERT,2,-,0.016129,-,0.016129,Vector
4,HKUDS,-,2,-,0.016129,0.016129,BM25
5,VECTOR_SEARCH,4,-,0.015625,-,0.015625,Vector
6,BM25,-,4,-,0.015625,0.015625,BM25


## 9. 결과 시각화

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fused = reciprocal_rank_fusion(vector_results, bm25_results, k=60)
docs_order = [r['id'] for r in fused]
vec_ranks = [vec_rank.get(d, len(vector_results)+2) for d in docs_order]
bm25_ranks = [bm25_rank.get(d, len(bm25_results)+2) for d in docs_order]
fused_ranks = list(range(1, len(docs_order)+1))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(docs_order))
width = 0.25

ax.bar(x - width, vec_ranks, width, label='Vector Rank', color='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
ax.bar(x, bm25_ranks, width, label='BM25 Rank', color='#FFCCBC', edgecolor='#E64A19', linewidth=1.5)
ax.bar(x + width, fused_ranks, width, label='RRF Fused Rank', color='#C8E6C9', edgecolor='#2E7D32', linewidth=1.5)

ax.set_xlabel('Entities', fontsize=12)
ax.set_ylabel('Rank (lower = better)', fontsize=12)
ax.set_title('Vector vs BM25 vs Hybrid(RRF) Ranking Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(docs_order, rotation=30, ha='right')
ax.legend(fontsize=10)
ax.invert_yaxis()
ax.set_ylim(max(max(vec_ranks), max(bm25_ranks)) + 0.5, 0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: images/ranking_comparison.png')

차트 저장: images/ranking_comparison.png


## 10. 설정 확인 및 정리

In [15]:
from lightrag.addon_params import default_addon_params

defaults = default_addon_params()
print('addon_params 기본값:')
for k, v in defaults.items():
    if k != 'chunker':
        print(f'  {k}: {v}')

print(f'\nenable_hybrid_search 기본값: {defaults.get("enable_hybrid_search")}')
assert defaults.get('enable_hybrid_search') == False
print('기본값이 False -> 기존 동작에 영향 없음 확인!')

# 스토리지 정리
await rag_vector.finalize_storages()
await rag_hybrid.finalize_storages()
print('\n스토리지 정리 완료')

INFO: Successfully finalized 12 storages


INFO: Successfully finalized 12 storages


addon_params 기본값:
  language: English
  entity_type_prompt_file: 
  enable_hybrid_search: False

enable_hybrid_search 기본값: False
기본값이 False -> 기존 동작에 영향 없음 확인!

스토리지 정리 완료


## 결론

### 비교 요약

| 쿼리 유형 | Vector Only | Hybrid (Vector + BM25) |
|---|---|---|
| 고유명사 ("GPT-4o") | 임베딩 근사 매칭 | **BM25 정확 매칭 보완** |
| 기관명 ("HKUDS") | 유사 의미 문서 혼동 가능 | **정확 토큰 매칭** |
| 기술 용어 ("BM25") | 유사 의미 문서 반환 | **해당 키워드 포함 문서 우선** |
| 서술형 쿼리 | 효과적 | 벡터 위주 + BM25 보조 |
| 복합 쿼리 | 의미 중심 | **키워드 + 의미 양쪽 활용** |

### 핵심 포인트

1. **BM25는 정확한 키워드 매칭에 강점** - 고유명사, 기술용어, 약어 등
2. **RRF는 양쪽 결과를 공정하게 병합** - 스코어 스케일 차이를 순위로 해결
3. **기존 동작 완벽 호환** - `enable_hybrid_search=False`가 기본값
4. **Lazy Invalidation** - 문서 삽입 시 자동 무효화, 다음 쿼리에서 재빌드

### LLM 엔드포인트 설정 정보

| 항목 | 값 |
|---|---|
| Base URL | `http://222.117.133.162:30010/v1` |
| 모델명 | `qwen-task-pool` |
| API Key | `asdf` |

위 엔드포인트는 OpenAI 호환 API (vLLM 서버)로, `openai_complete_if_cache` 함수를 통해 LightRAG와 연동됩니다.

---

## (보너스) 실제 LLM으로 문서 삽입 + 하이브리드 쿼리

아래 셀은 **LLM 엔드포인트에 접근 가능한 환경**에서 실행하세요.
`ainsert()`는 LLM을 호출하여 문서에서 자동으로 엔티티와 관계를 추출합니다.

```
Pipeline: 문서 텍스트 → LLM 엔티티/관계 추출 → VDB 임베딩 저장 → BM25 인덱스 빌드
```

In [ ]:
# === 실제 LLM으로 문서 삽입 + 하이브리드 쿼리 ===
# (LLM 엔드포인트 접근 가능 시에만 실행)

WORK_DIR_LLM = '/tmp/lightrag_demo_llm_hybrid'
if os.path.exists(WORK_DIR_LLM):
    shutil.rmtree(WORK_DIR_LLM)
os.makedirs(WORK_DIR_LLM)

# Hybrid 모드로 LightRAG 인스턴스 생성 (실제 LLM 사용)
rag_llm = LightRAG(
    working_dir=WORK_DIR_LLM,
    llm_model_func=llm_model_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMB_DIM,
        max_token_size=4096,
        func=mock_embedding,
    ),
    tokenizer=Tokenizer('simple-tokenizer', SimpleTokenizer()),
    addon_params={
        'enable_hybrid_search': True,
    },
)

await rag_llm.initialize_storages()

# 샘플 문서 (LLM이 엔티티/관계를 자동 추출)
SAMPLE_DOC = """
GPT-4o is a multimodal large language model developed by OpenAI in 2024. 
It can process text, images, and audio inputs simultaneously, achieving 
state-of-the-art performance on benchmarks like MMLU and HumanEval.

LightRAG is a graph-based Retrieval Augmented Generation system created by 
the HKUDS research group at the University of Hong Kong. Unlike traditional 
RAG systems that rely on flat chunk retrieval, LightRAG constructs knowledge 
graphs from documents and performs structured graph traversal for retrieval.

BM25 (Best Matching 25) is a probabilistic ranking function used in information 
retrieval systems like Elasticsearch and Apache Lucene. It scores documents 
based on term frequency (TF) and inverse document frequency (IDF), making it 
effective for exact keyword matching.

Hybrid search combines vector similarity search with keyword-based BM25 search 
using Reciprocal Rank Fusion (RRF). The RRF formula is: score(d) = sum(1/(k+rank)) 
where k=60. This approach leverages the strengths of both semantic understanding 
(vectors) and exact term matching (BM25).
"""

print('문서 삽입 중... (LLM이 엔티티/관계를 자동 추출합니다)')
try:
    await rag_llm.ainsert(SAMPLE_DOC)
    print('문서 삽입 완료!')
    
    # BM25 인덱스 확인
    await rag_llm._build_bm25_indices()
    print(f'\nBM25 인덱스 상태:')
    print(f'  청크: {len(rag_llm._bm25_chunks.corpus_ids) if rag_llm._bm25_chunks and rag_llm._bm25_chunks.is_built else 0}개')
    print(f'  엔티티: {len(rag_llm._bm25_entities.corpus_ids) if rag_llm._bm25_entities and rag_llm._bm25_entities.is_built else 0}개')
    print(f'  관계: {len(rag_llm._bm25_relations.corpus_ids) if rag_llm._bm25_relations and rag_llm._bm25_relations.is_built else 0}개')
    
    # 쿼리 테스트
    print('\n=== 하이브리드 쿼리 테스트 (실제 LLM) ===')
    for query in ['GPT-4o', 'HKUDS LightRAG', 'BM25 Elasticsearch']:
        print(f'\n쿼리: "{query}"')
        result = await rag_llm.aquery_data(query, param=QueryParam(mode='naive', top_k=3))
        if result.get('status') == 'success':
            chunks = result.get('data', {}).get('chunks', [])
            print(f'  검색된 청크: {len(chunks)}개')
            for c in chunks[:2]:
                print(f'  - {c.get("content", "")[:80]}...')
        else:
            print(f'  오류: {result.get("message", "unknown")}')
    
    await rag_llm.finalize_storages()
    
except Exception as e:
    print(f'\nLLM 연결 실패: {type(e).__name__}: {e}')
    print('위 셀의 ainsert_custom_kg 기반 데모 결과를 참고하세요.')
    await rag_llm.finalize_storages()